In [12]:
import os, json, re, time, getpass, random, threading
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================================
# STEP 1: Settings (Connect to the AI)
# ==========================================

In [13]:
HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "gemma4-26b"
TEMPERATURE = 0
SEED = 123

MAX_WORKERS = 2          # keep low; 8 overloaded the server
TIMEOUT = 180
MAX_RETRIES = 3
STOP_AFTER_ERRORS = 20

INPUT_EXCEL_FILE = "sample_5k_translated.xlsx"
CHECKPOINT_FILE  = "cold_annotation_checkpoint.jsonl"
OUTPUT_EXCEL     = "cold_annotated_5k.xlsx"
OUTPUT_JSON      = "cold_annotated_5k.json"

# columns produced by the model run (never part of the input)
RESULT_COLS = ["raw_output", "json_ok", "label", "explanation",
               "error", "tokens_prompt", "tokens_response", "seconds"]

API_KEY = os.environ.get("UNI_API_KEY") or getpass.getpass("Paste the Uni API key: ")
print("Key loaded:", bool(API_KEY))

Key loaded: True


# ==========================================
# STEP 2: The Annotation Prompt
# ==========================================

In [14]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

# Customize your labels and definition here
annotation_prompt = """

ROLE
You are a text annotator specializing in media content analysis and computational communication. Your task is to apply the Stereotype Content Model (SCM) to evaluate whether targeted social groups in translated German newspaper articles are portrayed with Coldness (the opposite of Warmth in the SCM).

TASK
Based on the multiple paragraphs provided, decide whether [Gruppe 1] (or an individual member of [Gruppe 1]) is attributed with coldness. The label is TRUE if the text contains coldness and FALSE if it does not. Prioritize explicit mentions of cold attributes. Implicit portrayals are also encouraged to be coded when the text clearly conveys coldness without using a trait word. Give a short explanation of why you chose TRUE or FALSE, based on the criteria below. While coding, target only [Gruppe 1]. Do NOT code whether other groups ([andere Gruppe]) display coldness toward [Gruppe 1].

CONTEXT
You are working with paragraphs from German newspaper articles from a wide range of the political spectrum. The paragraphs have been translated into English. Each input consists of several consecutive paragraphs from one article, together with the target group [Gruppe 1]. Return one label for the whole passage. Entity names in the text are masked as [Gruppe 1] and [andere Gruppe]. Keep these names exactly as written and do not translate, change or replace them, including in your explanation.

DEFINITION
Coldness in news coverage is defined as an individual's or group's perceived negative intention towards the perceiver or their group of belonging. Coldness is conceptualized through two dimensions in media reporting:

Unfriendliness/asociality: Framing the individual or group as unfriendly if they are depicted as someone who treats others in an unfriendly, rude and disrespectful manner.
Attribute markers include but are not limited to: unfriendly, cold, unsociable, aloof, unpleasant, hostile, rude, disliked

Immorality: Framing the individual or group as immoral if they are attributed as someone untrustworthy, dishonest, unfair and bad.
Attribute markers include but are not limited to: immoral, untrustworthy, dishonest, insincere, unfair, intolerant, selfish, malicious

CRITERIA: COLDNESS OCCURS IF

Case: A cold trait is attributed to [Gruppe 1] or a member.
Example: "The [Gruppe 1] are considered unpleasant / rude."

Case: A synonym or noun form of a cold trait is used.
Example: "Their dishonesty is shameful."

Case: A behaviour that clearly shows bad intentions toward others.
Example: "The [Gruppe 1] deliberately ruined their neighbor's business."

Case: A negative moral evaluation.
Example: "The [Gruppe 1] selfishly hoarded resources."

CRITERIA: COLDNESS DOES NOT OCCUR IF

Case: Others are cold TOWARD the group.
Example: "The local population cheated [Gruppe 1] out of their share." / "The public showed no willingness to assist [Gruppe 1]."
Why not: The coldness belongs to the Germans, not to [Gruppe 1].

Case: The group is only a victim.
Example: "Three [Gruppe 1] were attacked."
Why not: Being a victim is not a trait.

Case: It is about ability or courage (competence/incompetence).
Example: "The cowardly [Gruppe 1]", "lazy"
Why not: Courage, diligence and skill, including their antonyms, = competence/incompetence.

Case: The cold word describes someone else.
Example: "Filip is rude to everyone," says the [Gruppe 1].
Why not: [Gruppe 1] is only the speaker.

Case: It is only a threat frame.
Example: "[Gruppe 1] weakens the economy."
Why not: An economic disadvantage, not a trait (Frames dimension).

Case: No portrayal at all.
Example: "In 2022, 12,000 [Gruppe 1] applied for asylum."
Why not: Facts and statistics only.

DECISION RULES
tba.

OUTPUT FORMAT
Return only a valid JSON object, with no text before or after it. The object has exactly two keys, in this order:
"explanation": 1-2 sentences explaining the label, citing the specific words or behavior that led to it.
"label": the string "TRUE" or "FALSE".

Example of valid output:
{"explanation": "The text calls the [Gruppe 1] rude and hostile toward their neighbors.", "label": "TRUE"}

"""

instructions = annotation_prompt
reminder = None  # this is just an additonal vairable to nudge the llm to a specific direction
JSON_MODE = True

print("--- Instructions to annotate COLD Loaded ---")
print(instructions)

--- Instructions to annotate COLD Loaded ---


ROLE
You are a text annotator specializing in media content analysis and computational communication. Your task is to apply the Stereotype Content Model (SCM) to evaluate whether targeted social groups in translated German newspaper articles are portrayed with Coldness (the opposite of Warmth in the SCM).

TASK
Based on the multiple paragraphs provided, decide whether [Gruppe 1] (or an individual member of [Gruppe 1]) is attributed with coldness. The label is TRUE if the text contains coldness and FALSE if it does not. Prioritize explicit mentions of cold attributes. Implicit portrayals are also encouraged to be coded when the text clearly conveys coldness without using a trait word. Give a short explanation of why you chose TRUE or FALSE, based on the criteria below. While coding, target only [Gruppe 1]. Do NOT code whether other groups ([andere Gruppe]) display coldness toward [Gruppe 1].

CONTEXT
You are working with paragraphs from Ger

# ==========================================
# STEP 3: Paragraph Input Data
# ==========================================

In [15]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

try:
    sample = pd.read_excel(INPUT_EXCEL_FILE)
except FileNotFoundError:
    raise FileNotFoundError(f"Could not find '{INPUT_EXCEL_FILE}' in {os.getcwd()}")

if "row_id" not in sample.columns:
    sample.insert(0, "row_id", range(len(sample)))

if "translation" not in sample.columns:
    raise KeyError(f"No 'translation' column. Columns found: {sample.columns.tolist()}")

empty = sample["translation"].isna() | (sample["translation"].astype(str).str.strip() == "")
if empty.any():
    print(f"Dropping {int(empty.sum())} rows with empty translation "
          f"(row_ids kept, first 20: {sample.loc[empty, 'row_id'].tolist()[:20]})")
    sample = sample.loc[~empty].copy()

assert sample["row_id"].is_unique, "row_id must be unique"
print(f"Loaded {len(sample)} paragraph(s) from '{INPUT_EXCEL_FILE}'.")
print("Columns:", sample.columns.tolist())

Dropping 24 rows with empty translation (row_ids kept, first 20: [780, 1111, 1491, 2542, 2564, 2743, 3129, 3130, 3144, 3384, 3388, 3634, 3678, 3827, 3852, 4108, 4469, 4662, 4721, 4756])
Loaded 4976 paragraph(s) from 'sample_5k_translated.xlsx'.
Columns: ['row_id', 'article_id', 'pub', 'par_index', 'text_block', 'group', 'length', 'translation', 'raw_output', 'json_ok', 'placeholder_ok', 'length_ratio', 'length_ok', 'tokens_prompt', 'tokens_response', 'seconds', 'error', 'model', 'seed']


# ==========================================
# STEP 4: The LLM API Call Function
# ==========================================

In [16]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

def call_llm(text, instructions, reminder=None, json_mode=True, max_retries=MAX_RETRIES):
    user_msg = f"Text: '{text}'"
    if reminder:
        user_msg += f"\nReminder: {reminder}"

    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": instructions},
            {"role": "user", "content": user_msg},
        ],
        "temperature": TEMPERATURE,
        "seed": SEED,
        "reasoning_effort": "none",
        "max_tokens": 1000,
    }
    if json_mode:
        body["response_format"] = {"type": "json_object"}

    last_error = None
    for attempt in range(1, max_retries + 1):
        resp = None
        try:
            resp = requests.post(f"{HOST}/chat/completions",
                                 headers={"Authorization": f"Bearer {API_KEY}"},
                                 json=body, timeout=TIMEOUT)
            resp.raise_for_status()
            j = resp.json()
            return {
                "output": j["choices"][0]["message"]["content"].strip(),
                "tokens_prompt": j["usage"]["prompt_tokens"],
                "tokens_response": j["usage"]["completion_tokens"],
                "error": None,
            }
        except Exception as e:
            last_error = str(e)
            if resp is None:
                resp = getattr(e, "response", None)
            status = resp.status_code if resp is not None else None
            print(f"Attempt {attempt}/{max_retries} failed (status {status}): {e}")

            # permanent client errors: don't retry (408 and 429 are retryable)
            if status is not None and 400 <= status < 500 and status not in (408, 429):
                break

            if attempt < max_retries:
                if status == 429:
                    try:
                        wait = float(resp.headers.get("Retry-After"))
                    except (TypeError, ValueError):
                        wait = 60
                elif status == 504:
                    wait = 15 * (2 ** (attempt - 1))
                else:
                    wait = 5 * attempt
                time.sleep(wait + random.uniform(0, 1))

    return {"output": None, "tokens_prompt": None, "tokens_response": None, "error": last_error}


def parse_llm_json(raw):
    """Strip code fences, fall back to the first {...} block. Returns dict or None."""
    if not raw or not isinstance(raw, str):
        return None
    s = re.sub(r"^```(?:json)?\s*", "", raw.strip())
    s = re.sub(r"\s*```$", "", s)
    candidates = [s]
    m = re.search(r"\{.*\}", s, re.S)
    if m:
        candidates.append(m.group(0))
    for c in candidates:
        try:
            d = json.loads(c)
            if isinstance(d, dict):
                return d
        except Exception:
            pass
    return None


def load_checkpoint(path):
    """Last line per row_id wins, so retried failures overwrite earlier failures."""
    cache = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    d = json.loads(line)
                    cache[d["row_id"]] = d
                except Exception:
                    continue
    return cache


def is_done(d):
    return d.get("error") is None and d.get("label") in ("TRUE", "FALSE")

# ==========================================
# STEP 5: Run Annotation Loop 
# ==========================================

In [17]:
if "API_KEY" not in globals() or "sample" not in globals() or "instructions" not in globals():
    raise RuntimeError("Please run STEPS 1-4 first.")

cache = load_checkpoint(CHECKPOINT_FILE)
done = {rid for rid, d in cache.items() if is_done(d)}
todo_ids = [rid for rid in sample["row_id"] if rid not in done]
print(f"Resumed: {len(done)} valid rows in checkpoint, {len(todo_ids)} to do.")

file_lock = threading.Lock()
stop_flag = threading.Event()

def process_row(row):
    rid = row["row_id"]
    rid = rid.item() if hasattr(rid, "item") else rid
    if rid in done or stop_flag.is_set():
        return None

    start = time.time()
    resp = call_llm(str(row["translation"]), instructions, reminder, json_mode=JSON_MODE)
    parsed = parse_llm_json(resp["output"])
    label = parsed.get("label") if parsed else None
    label = None if label is None else str(label).strip().upper()

    entry = {
        "row_id": rid,
        "raw_output": resp["output"],
        "json_ok": parsed is not None,
        "label": label,
        "explanation": parsed.get("explanation") if parsed else None,
        "error": resp["error"],
        "tokens_prompt": resp["tokens_prompt"],
        "tokens_response": resp["tokens_response"],
        "seconds": round(time.time() - start, 1),
    }
    with file_lock:
        with open(CHECKPOINT_FILE, "a", encoding="utf-8") as cp:
            cp.write(json.dumps(entry, ensure_ascii=False) + "\n")
    return entry

todo_rows = [row for _, row in sample.iterrows() if row["row_id"] not in done]
errors_in_a_row, processed = 0, 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(process_row, row) for row in todo_rows]
    for fut in as_completed(futures):
        entry = fut.result()
        if entry is None:
            continue
        processed += 1
        errors_in_a_row = errors_in_a_row + 1 if not is_done(entry) else 0
        if errors_in_a_row >= STOP_AFTER_ERRORS and not stop_flag.is_set():
            stop_flag.set()
            print(f"Stopping: {STOP_AFTER_ERRORS} failures in a row. Re-run this cell to resume.")
        if processed % 50 == 0:
            print(f"Progress: {processed}/{len(todo_rows)} processed this run.")

cache = load_checkpoint(CHECKPOINT_FILE)
n_valid = sum(1 for rid in sample["row_id"] if rid in cache and is_done(cache[rid]))
print(f"\nDone. Valid annotations: {n_valid}/{len(sample)}. "
      f"{'All good, go to STEP 6.' if n_valid == len(sample) else 'Re-run this cell to retry the rest.'}")

Resumed: 5000 valid rows in checkpoint, 0 to do.

Done. Valid annotations: 4976/4976. All good, go to STEP 6.


# ==========================================
# STEP 6: Quality Checks & Verification
# ==========================================

In [18]:
if "sample" not in globals():
    raise RuntimeError("Please run STEPS 1-4 first.")

cache = load_checkpoint(CHECKPOINT_FILE)
if not cache:
    raise RuntimeError(f"Checkpoint '{CHECKPOINT_FILE}' is empty or missing (cwd: {os.getcwd()}).")

ck_df = pd.DataFrame(cache.values())
for c in RESULT_COLS:
    if c not in ck_df.columns:
        ck_df[c] = None
ck_df = ck_df[["row_id"] + RESULT_COLS]

# drop stale result columns from sample so the merge can't create _x/_y duplicates
stale = [c for c in RESULT_COLS if c in sample.columns]
if stale:
    print("Dropped stale result columns from sample:", stale)
base = sample.drop(columns=stale)

final_df = base.merge(ck_df, on="row_id", how="left", validate="one_to_one")

# recompute json_ok from raw_output so old and new checkpoint lines are treated alike
final_df["json_ok"] = final_df["raw_output"].apply(
    lambda r: isinstance(r, str) and parse_llm_json(r) is not None)

# ---------- quality checks ----------
exp = final_df["explanation"].fillna("").astype(str)
src = final_df["translation"].astype(str)
allowed_tags = {"[Gruppe 1]", "[andere Gruppe]"}

bad_json  = ~final_df["json_ok"]
bad_label = ~final_df["label"].isin(["TRUE", "FALSE"])
bad_exp   = exp.str.strip() == ""
bad_case  = exp.apply(lambda s: any(t not in allowed_tags
                                    for t in re.findall(r"\[[^\]]*[Gg]ruppe[^\]]*\]", s)))
bad_mask  = src.str.contains("[Gruppe 1]", regex=False) & ~exp.str.contains("[Gruppe 1]", regex=False)

print(f"Rows: {len(final_df)}")
print(f"  JSON not parseable:                  {int(bad_json.sum())}")
print(f"  Label not TRUE/FALSE:                {int(bad_label.sum())}")
print(f"  Empty explanation:                   {int(bad_exp.sum())}")
print(f"  Wrong mask casing in explanation:    {int(bad_case.sum())}   (e.g. [Andere Gruppe])")
print(f"  [Gruppe 1] not mentioned (soft):     {int(bad_mask.sum())}   (can be legitimate)")

hard = bad_json | bad_label | bad_exp | bad_case
print(f"\nRows with at least one hard issue: {int(hard.sum())}")
if hard.any():
    print(final_df.loc[hard, ["row_id", "label", "explanation", "error"]].head(20).to_string())

print("\nLabel distribution:")
print(final_df["label"].value_counts(dropna=False))

Dropped stale result columns from sample: ['raw_output', 'json_ok', 'error', 'tokens_prompt', 'tokens_response', 'seconds']
Rows: 4976
  JSON not parseable:                  0
  Label not TRUE/FALSE:                0
  Empty explanation:                   0
  Wrong mask casing in explanation:    169   (e.g. [Andere Gruppe])
  [Gruppe 1] not mentioned (soft):     71   (can be legitimate)

Rows with at least one hard issue: 169
     row_id  label                                                                                                                                                                                                                                                                                                            explanation error
6         6  FALSE                                                                                                                                              The text discusses the recruitment of [Gruppe 1] and the integration of [A

# ==========================================
# STEP 7: Save the Results
# ==========================================

In [19]:
if "final_df" not in globals():
    raise RuntimeError("Please run STEP 6 first.")
assert final_df["label"].isin(["TRUE", "FALSE"]).any(), "final_df has no valid labels, check STEP 6 output."

final_df.to_excel(OUTPUT_EXCEL, index=False)
final_df.to_json(OUTPUT_JSON, orient="records", indent=2, force_ascii=False)

# read back to confirm the files really contain the labels
check = pd.read_excel(OUTPUT_EXCEL)
print("Saved:", OUTPUT_EXCEL, "and", OUTPUT_JSON)
print("Columns in saved Excel:", check.columns.tolist())
print(check["label"].value_counts(dropna=False))

Saved: cold_annotated_5k.xlsx and cold_annotated_5k.json
Columns in saved Excel: ['row_id', 'article_id', 'pub', 'par_index', 'text_block', 'group', 'length', 'translation', 'placeholder_ok', 'length_ratio', 'length_ok', 'model', 'seed', 'raw_output', 'json_ok', 'label', 'explanation', 'error', 'tokens_prompt', 'tokens_response', 'seconds']
label
False    4541
True      435
Name: count, dtype: int64
